In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/7817_1.csv")

print(df.columns)

reviews = df['reviews.text'].dropna().tolist()

print("Total reviews:", len(reviews))

Index(['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded',
       'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer',
       'manufacturerNumber', 'name', 'prices', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes',
       'upc', 'weight'],
      dtype='object')
Total reviews: 1597


In [ ]:
!pip install pandas sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.0 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(reviews, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype('float32')

dimension = embeddings.shape[1]
index = faiss.IndexHNSWFlat(dimension, 32)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 1597


In [ ]:
def semantic_search(query, top_k=5):
    query_vector = model.encode([query])
    query_vector = np.array(query_vector).astype('float32')

    faiss.normalize_L2(query_vector)

    distances, indices = index.search(query_vector, top_k)

    results = []
    for idx in indices[0]:
        results.append(reviews[idx])

    return results

In [ ]:
query = "battery drains fast"

results = semantic_search(query)

for i, r in enumerate(results):
    print(f"\nResult {i+1}: {r}")


Result 1: First off,i appreciate you reading this review :) Just so you get a little background on me, we are major tech users. We have multiple tablets from apple,samsung,and asus. This fire tablet is comparable in a different way. This tablet uses a amazon based operating system, Fire os. So no play store or itunes... As a prime member however, this tablet is an incredible value because you get instant access to all of your prime benifits. Ive had a chance to really put this product thru its paces over the past 2 days. I ordered 2 of these tablets so me and my wife could really put these to use. We have Gone thru 3 full power cycles (fully charging and fully draining the battery thru use) it does get 10-11 hours of hefty use so battery life is a plus! This tablet is a keeper. Screen looks fantastic. Colors seem to be just slightly dull however the resoloution for everyday use is perfect for this price point. Speed is snappy enough for basic use and handles more complex tasks and gam

In [ ]:
def keyword_search(query, top_k=5):
    results = []

    for review in reviews:
        if query.lower() in review.lower():
            results.append(review)

        if len(results) >= top_k:
            break

    return results

In [ ]:
query = "battery drains fast"

print("Keyword Search Results:")
for r in keyword_search(query):
    print("-", r)

print("\n Vector Search Results:")
for r in semantic_search(query):
    print("-", r)

Keyword Search Results:
- ..exactly what I wanted, but the battery drains FAST when on hands free mode.

 Vector Search Results:
- First off,i appreciate you reading this review :) Just so you get a little background on me, we are major tech users. We have multiple tablets from apple,samsung,and asus. This fire tablet is comparable in a different way. This tablet uses a amazon based operating system, Fire os. So no play store or itunes... As a prime member however, this tablet is an incredible value because you get instant access to all of your prime benifits. Ive had a chance to really put this product thru its paces over the past 2 days. I ordered 2 of these tablets so me and my wife could really put these to use. We have Gone thru 3 full power cycles (fully charging and fully draining the battery thru use) it does get 10-11 hours of hefty use so battery life is a plus! This tablet is a keeper. Screen looks fantastic. Colors seem to be just slightly dull however the resoloution for e